In [8]:
import os
import gc
import torch
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
import os
os.chdir("/Users/zengjian/PyTorch/MLP/data/")

sample_filter = 20
var_filter = 10
chunk_size = 2000
batch_size = 64
seed = 42

X_file = "expression.tsv"
y_file = "merged_sample_quality_annotations.tsv"
def load_and_preprocess_tcga_data(X_file,y_file):
    
    # 1. 去除重复y label
    print("remove duplicates y label sample")
    df_y = pd.read_csv(y_file, sep="\t", usecols=['aliquot_barcode', 'cancer type'])
    df_y.drop_duplicates(subset=['aliquot_barcode'], keep='first', inplace=True)
    df_y.set_index('aliquot_barcode', inplace=True)

    # 过滤掉cancer type中样本低于20个样本的type
    print("filter cancer type with samples lower than 20")
    class_counts = df_y['cancer type'].value_counts()
    valid_classes = class_counts[class_counts >= sample_filter].index
    df_y = df_y[df_y['cancer type'].isin(valid_classes)]
    valid_samples = set(df_y.index)
    
    # 2. 提取样本名（读取的原始数据中列名为样本，行名为基因）
    print("keep X samples in y data")
    x_header = pd.read_csv(X_file, sep="\t", nrows=0).columns.tolist()
    gene_col = x_header[0]
    target_cols = [gene_col] + list(set(x_header[1:]).intersection(valid_samples))
    
    # 3. Chunk分块读取表达矩阵，过滤方差小于10的特征（gene）
    print(f"Reading expression of {chunk_size}, and gene variance filtering < {var_filter}")
    chunks = []
    reader = pd.read_csv(X_file, sep="\t", index_col=0, usecols=target_cols, chunksize=chunk_size)
    
    for chunk in reader:
        chunk.dropna(axis=0, how='any', inplace=True)
        if not chunk.empty:
            # 方差过滤
            chunk = chunk[chunk.var(axis=1) > var_filter]
        if not chunk.empty:
            chunks.append(chunk.T)
    #删除reader,释放内存
    del reader
    gc.collect()

    # 4. X_file与y_label合并
    X_final = pd.concat(chunks, axis=1)
    
    del chunks
    gc.collect()

    X_final = X_final[~X_final.index.duplicated(keep='first')]
    y_final = df_y.loc[X_final.index]
    ##给字符型cancer type编码成数字
    le = LabelEncoder()
    y_encoded = le.fit_transform(y_final['cancer type'])

    # 5. 标准化
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_final)

    # 6. 分区划分 (Train: 70%, Valid: 15%, Test: 15%)
    X_train, X_temp, y_train, y_temp = train_test_split(
        X_scaled, y_encoded, test_size=0.30, random_state=seed, stratify=y_encoded
    )
    X_valid, X_test, y_valid, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, random_state=seed, stratify=y_temp
    )

    print(f"Total Sample and Feature (Gene): {X_final.shape}")
    print(f"Train_size: {X_train.shape[0]}, Valid_size: {X_valid.shape[0]}, Test_size: {X_test.shape[0]}")
    print(f"Target cancer types: {len(le.classes_)}")

    return X_train, y_train, X_valid, y_valid, X_test, y_test

if __name__ == "__main__":
    # 数据预处理
    X_train, y_train, X_valid, y_valid, X_test, y_test = load_and_preprocess_tcga_data(X_file = X_file,
                                                                                       y_file = y_file)
    
    # 转换为 PyTorch DataLoaders
    train_loader = DataLoader(
        TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long)), 
        batch_size=batch_size, shuffle=True
    )
    valid_loader = DataLoader(
        TensorDataset(torch.tensor(X_valid, dtype=torch.float32), torch.tensor(y_valid, dtype=torch.long)), 
        batch_size=batch_size, shuffle=False
    )
    test_loader = DataLoader(
        TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long)), 
        batch_size=batch_size, shuffle=False
    )

remove duplicates y label sample
filter cancer type with samples lower than 20
keep X samples in y data
Reading expression of 2000, and gene variance filtering
Total Sample and Feature (Gene): (11069, 16278)
Train_size: 7748, Valid_size: 1660, Test_size: 1661
Target cancer types: 33


In [ ]:
# 变量保存给下一个module使用
processed_data = {
        'train_loader': train_loader,
        'valid_loader': valid_loader,
        'test_loader': test_loader,
        'y_train':y_train,
        'input_dim': X_train.shape[1],
        'num_classes': len(np.unique(y_train)) 
    }
    
    # 保存为 .pt 文件（速度极快，且完美保留 Tensor 属性）
torch.save(processed_data, 'PyTorch/MLP/data/tcga_processed_tensors.pt')